# 26. Paraphrase smoothing at test time

The adversary wins by rewriting the injection into prose the detector reads as benign. The defender can rewrite too. This notebook scores each document as the maximum over its own score and the scores of k paraphrases generated by the defender's model (Qwen2.5-3B-Instruct in fp16, not the attacker's 7B). An attacker must now find a rewrite whose every paraphrase also evades, and paraphrasing tends to put the imperative back.

Three numbers per detector, before and after smoothing: the static miss rate on the 91 original injections; the miss rate when the attacker moves first (selects against the base detector, then meets the smoothed one); and the miss rate when the attacker moves second (selects against the smoothed score itself, which is the only fair test). Thresholds are set on a 150-host subset of BIPIA benign documents that is paraphrased the same way, so smoothing pays for its own false positives; a 100-prompt NotInject subset checks over-defense. The independence prediction e to the power k is compared with the observed evasion rather than assumed.

Functionality of the newly selected rewrites is not judged here; if smoothing moves the miss rate, that follows in the next notebook. Cheap on purpose: about 2,800 short generations and scoring.

**Outputs.** `data/smooth/para_*.json` (paraphrases), `data/smooth/score_*.npy`, `reports/smoothing_eval.csv`, `reports/smoothing_ek.csv`, `figures/smoothing.png`.

In [ ]:
# === SESSION BOOTSTRAP (run first): install from the local disk, then mount ===
!pip install -q transformers datasets accelerate sentencepiece bitsandbytes
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/PICALIB_Research'
REPO_DIR   = os.path.join(DRIVE_ROOT, 'picalib-research')
!git config --global user.name  "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git config --global credential.helper "store --file={DRIVE_ROOT}/.git-credentials"
%cd {REPO_DIR}
sys.path.insert(0, 'src')
!git pull
os.makedirs('reports', exist_ok=True); os.makedirs('figures', exist_ok=True); os.makedirs('data/smooth', exist_ok=True)
import torch, gc; print('Session ready. GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

## Stage A. Module and inputs

In [ ]:
%%writefile src/smoothing.py
"""Paraphrase smoothing at test time: score a document as the maximum over the base score and the scores of k
paraphrases the defender generates with its own model. An attacker who rewrote an injection to evade the base
detector must now also evade every paraphrase of the rewritten document, and paraphrasing tends to restore the
imperative form the detector keys on. Evaluated with the attacker moving second (selecting against the smoothed
score), which is the only fair test."""
from __future__ import annotations
import numpy as np

DEF_SYS = ("You rewrite a document in different words. Keep every fact and every request or instruction the document "
           "contains, in the same order, addressed to the same reader. Change the wording and sentence structure "
           "throughout. Output only the rewritten document, nothing else.")


def _budget(doc):
    w = len(doc.split()); return 260 if w <= 120 else (480 if w <= 260 else 768)


def paraphrase_docs(docs, gen, k=4, temperature=0.8, batch_size=8):
    """gen(prompts, system, max_new_tokens, temperature, batch_size, seed) -> list[str]. Returns list of k-lists."""
    out = [[] for _ in docs]
    buckets = {}
    for i, d in enumerate(docs): buckets.setdefault(_budget(d), []).append(i)
    for j in range(k):
        for mx, idx in buckets.items():
            gens = gen([docs[i] for i in idx], DEF_SYS, mx, temperature, batch_size, 1000 + j)
            for i, g in zip(idx, gens):
                g = (g or "").strip()
                out[i].append(g if len(g) >= 20 else docs[i])   # a failed generation falls back to the document itself
    return out


def smooth(base, para, how="max"):
    """base: (n,), para: (n,k) -> (n,) smoothed scores."""
    base = np.asarray(base, float); para = np.asarray(para, float)
    allsc = np.concatenate([base[:, None], para], axis=1)
    return allsc.max(1) if how == "max" else allsc.mean(1)


def group_min_index(scores, owner):
    scores = np.asarray(scores); n = int(owner.max()) + 1
    return np.array([np.where(owner == i)[0][int(np.argmin(scores[owner == i]))] for i in range(n)])


def ek_analysis(base_adv, para_adv, t):
    """Per-variant: observed evasion under max vs. the independence prediction (base < t) * e^k."""
    base_adv = np.asarray(base_adv); para_adv = np.asarray(para_adv); k = para_adv.shape[1]
    e = (para_adv < t).mean(1); obs = ((base_adv < t) & (para_adv < t).all(1)).astype(float); pred = (base_adv < t) * e ** k
    return dict(k=k, mean_per_paraphrase_evasion=float(e.mean()), observed_evasion=float(obs.mean()), independence_prediction=float(pred.mean()),
                base_evasion=float((base_adv < t).mean()))


In [ ]:
import importlib, json, numpy as np, pandas as pd
os.makedirs('data/smooth', exist_ok=True)
import smoothing, phase5_redo, interval_panel, data_loaders; [importlib.reload(m) for m in (smoothing, phase5_redo, interval_panel, data_loaders)]
from smoothing import paraphrase_docs, smooth, group_min_index, ek_analysis
from phase5_redo import flat_variants
from interval_panel import three_thresholds, fnr, fpr, KEYS
from data_loaders import load_bipia_categorized, load_notinject, BIPIA_MALICIOUS
SUB = pd.DataFrame(json.load(open('data/phase5v2/subset.json'))); REW = json.load(open('data/phase5v2/rewrites.json'))
VAR, OWNER, FIRST = flat_variants(SUB, REW); print('adversarial variants:', len(VAR), 'from', len(SUB), 'injections')
bip = load_bipia_categorized(per_category=30, categories=BIPIA_MALICIOUS); hosts_all = bip[bip.label == 0].text.tolist()
rs = np.random.RandomState(0); HOSTS = [hosts_all[i] for i in sorted(rs.choice(len(hosts_all), 150, replace=False))]
noti = load_notinject(); NOTI = [noti.text.iloc[i] for i in sorted(np.random.RandomState(1).choice(len(noti), 100, replace=False))]
SETS = {'adv': VAR, 'hosts': HOSTS, 'notinject': NOTI}; print({k: len(v) for k, v in SETS.items()})

## Stage B. Defender paraphrases (cached; the only slow stage)

In [ ]:
from targets import load_lm, generate_batch
K = 4
PARA = {}
need = [s for s in SETS if not os.path.exists(f'data/smooth/para_{s}.json')]
if need:
    rm, rt = load_lm('Qwen/Qwen2.5-3B-Instruct', four_bit=False)
    def gen(prompts, system, mx, temp, bs, seed):
        torch.manual_seed(seed); return generate_batch(rm, rt, prompts, system=system, max_new_tokens=mx, batch_size=bs, temperature=temp)
    for s in need:
        print('paraphrasing', s, len(SETS[s]), 'documents x', K); PARA[s] = paraphrase_docs(SETS[s], gen, k=K); json.dump(PARA[s], open(f'data/smooth/para_{s}.json', 'w'))
    del rm; gc.collect(); torch.cuda.empty_cache()
for s in SETS: PARA[s] = json.load(open(f'data/smooth/para_{s}.json'))
print('example (adversarial variant 1, paraphrase 1):\n', PARA['adv'][1][0][:400])
print('\nfallbacks (paraphrase identical to source):', {s: int(sum(p == d for d, ps in zip(SETS[s], PARA[s]) for p in ps)) for s in SETS})

## Stage C. Score base documents and paraphrases (cached)

In [ ]:
from structural_train import score_texts
from detectors import score_released
DETS = {'structural (full)': ('struct', 'data/structural/model_full/best'), 'structural (full_aug)': ('struct', 'data/structural/model_full_aug/best'),
        'ProtectAI-v2': ('rel', 'protectai/deberta-v3-base-prompt-injection-v2'), 'Prompt-Guard-2 (86M)': ('rel', 'meta-llama/Llama-Prompt-Guard-2-86M')}
def tag(n): return n.replace(' ', '_').replace('(', '').replace(')', '')
if any(not os.path.exists(f"data/smooth/score_{tag(n)}_{s}_para.npy") for n in DETS if DETS[n][0] == 'rel' for s in SETS):
    from huggingface_hub import login; import getpass
    _t = getpass.getpass('HF token (gated Prompt-Guard checkpoint): ').strip()
    if _t: login(token=_t)
S_BASE, S_PARA = {}, {}
for name, (kind, mid) in DETS.items():
    S_BASE[name], S_PARA[name] = {}, {}
    for s, docs in SETS.items():
        fb, fp = f'data/smooth/score_{tag(name)}_{s}_base.npy', f'data/smooth/score_{tag(name)}_{s}_para.npy'
        if not (os.path.exists(fb) and os.path.exists(fp)):
            flat = [p for ps in PARA[s] for p in ps]; print('scoring', name, s, len(docs), '+', len(flat))
            sc = score_texts(mid, docs + flat) if kind == 'struct' else score_released(docs + flat, mid)
            np.save(fb, sc[:len(docs)]); np.save(fp, sc[len(docs):].reshape(len(docs), K))
        S_BASE[name][s] = np.load(fb); S_PARA[name][s] = np.load(fp)
print('scored:', list(DETS))

## Stage D. Before and after smoothing, attacker first and attacker second

In [ ]:
rows, ekrows = [], []
for name in DETS:
    for how in ('base', 'max', 'mean'):
        agg = (lambda s: S_BASE[name][s]) if how == 'base' else (lambda s, h=how: smooth(S_BASE[name][s], S_PARA[name][s], h))
        H = agg('hosts'); T = three_thresholds(H); A = agg('adv'); N = agg('notinject')
        sel_first = group_min_index(S_BASE[name]['adv'], OWNER)        # attacker chose against the base detector
        sel_second = group_min_index(A, OWNER)                           # attacker chooses against the smoothed score
        r = dict(detector=name, aggregation=how, k=K)
        for key in KEYS:
            t = T[key]
            r[f'static_FNR_{key}'] = fnr(A[FIRST], t); r[f'adaptive_first_FNR_{key}'] = fnr(A[sel_first], t); r[f'adaptive_second_FNR_{key}'] = fnr(A[sel_second], t)
            r[f'hosts_FPR_{key}'] = fpr(H, t); r[f'notinject_FPR_{key}'] = fpr(N, t)
        rows.append(r)
        if how == 'max':
            ekrows.append(dict(detector=name, threshold='guar', **ek_analysis(S_BASE[name]['adv'][sel_second], S_PARA[name]['adv'][sel_second], T['guar'])))
EV = pd.DataFrame(rows); EV.to_csv('reports/smoothing_eval.csv', index=False)
EK = pd.DataFrame(ekrows); EK.to_csv('reports/smoothing_ek.csv', index=False)
pd.set_option('display.width', 250)
print('=== miss rates at the guaranteed 1 percent host-FPR threshold (150-host calibration) ===')
print(EV[['detector', 'aggregation', 'static_FNR_guar', 'adaptive_first_FNR_guar', 'adaptive_second_FNR_guar', 'hosts_FPR_guar', 'notinject_FPR_guar']].round(3).to_string(index=False))
print('\n=== at the low end of the interval (more permissive threshold) ===')
print(EV[['detector', 'aggregation', 'static_FNR_lo', 'adaptive_second_FNR_lo', 'hosts_FPR_lo', 'notinject_FPR_lo']].round(3).to_string(index=False))
print('\n=== independence check on the attacker-second selection: observed evasion vs e^k ==='); print(EK.round(3).to_string(index=False))

## Stage E. Figure

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
piv = EV[EV.aggregation.isin(['base', 'max'])].pivot_table(index='detector', columns='aggregation', values=['static_FNR_guar', 'adaptive_second_FNR_guar'])
x = np.arange(len(piv)); w = 0.2
for j, (m, a, lab) in enumerate([('static_FNR_guar', 'base', 'static, base'), ('static_FNR_guar', 'max', 'static, smoothed'), ('adaptive_second_FNR_guar', 'base', 'adaptive, base'), ('adaptive_second_FNR_guar', 'max', 'adaptive (attacker second), smoothed')]):
    axes[0].bar(x + (j - 1.5) * w, piv[(m, a)].values, w, label=lab)
axes[0].set_xticks(x); axes[0].set_xticklabels(piv.index, rotation=20, fontsize=8); axes[0].set_ylim(0, 1.05); axes[0].set_ylabel('miss rate at guaranteed 1 percent host FPR'); axes[0].set_title(f'Paraphrase smoothing, k = {K}'); axes[0].legend(fontsize=7); axes[0].grid(axis='y', alpha=.3)
axes[1].bar(np.arange(len(EK)) - 0.15, EK.observed_evasion, 0.3, label='observed evasion (smoothed, attacker second)'); axes[1].bar(np.arange(len(EK)) + 0.15, EK.independence_prediction, 0.3, label='independence prediction e^k')
axes[1].set_xticks(np.arange(len(EK))); axes[1].set_xticklabels(EK.detector, rotation=20, fontsize=8); axes[1].set_ylim(0, 1.05); axes[1].set_title('Does e^k hold?'); axes[1].legend(fontsize=7); axes[1].grid(axis='y', alpha=.3)
plt.tight_layout(); plt.savefig('figures/smoothing.png', dpi=150, bbox_inches='tight'); plt.show()

## Stage F. Log and commit

In [ ]:
from reslog import log_result
summary = ('Paraphrase smoothing (k=%d), guaranteed-threshold miss rates:\n' % K) + EV[['detector', 'aggregation', 'static_FNR_guar', 'adaptive_first_FNR_guar', 'adaptive_second_FNR_guar', 'hosts_FPR_guar', 'notinject_FPR_guar']].round(3).to_string(index=False) + '\n\nIndependence check:\n' + EK.round(3).to_string(index=False)
log_result('nb26: paraphrase smoothing at test time, attacker first and second', summary, csv_df=EV, csv_name='smoothing_eval.csv')
print(summary)

# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
r = subprocess.run(["python", "tools/commit_cell.py", "nb26: paraphrase smoothing at test time (k=4, defender 3B paraphraser), attacker-first and attacker-second evaluation, e^k check; add src/smoothing.py"], capture_output=True, text=True)
print(r.stdout); print(r.stderr)